# 03. Enformer — 서열에서 유전체 트랙 예측하기 (GAPDH TSS 주변)

**등급**: 🟢 Mac 로컬(MPS/CPU) · **별도 환경 필요**: `.venv-enformer` (이유는 아래) · **다운로드**: Enformer 가중치 약 1.0 GB(CC BY 4.0), 트랙 메타데이터 0.8 MB, UCSC REST API 조회 몇 번

> ⚠️ **환경**: `enformer-pytorch`는 `transformers==4.56.2`로 고정한다. 이 저장소의 기본 `.venv`(transformers 5.x)에 설치하면 다른 노트북이 깨질 수 있어 **격리 환경**에서 실행한다.
> `uv venv --python 3.12 .venv-enformer && uv pip install --python .venv-enformer/bin/python -r requirements/enformer.txt` 후 이 노트북의 커널을 그 환경으로 선택한다.

**질문**: 지도학습된 sequence-to-function 모델은 서열만으로 **유전자 TSS 주변의 CAGE·DNase·히스톤 신호**를 예측하는가? 서열을 망가뜨리면 사라지는가? TSS 근처 염기 변화가 먼 곳의 염기 변화보다 예측을 더 크게 바꾸는가?

**모델**: Enformer — 입력 196,608 bp, 출력은 **중앙 896 bins(128 bp 단위, 약 114,688 bp)** × 사람 5,313 트랙 (DNase/ATAC, 히스톤·TF ChIP, CAGE).
이것은 DNA 언어모델이 아니라 **실험 트랙을 직접 예측하도록 지도학습**된 모델이다 (강의노트 2.2).

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


def find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "SETUP.md").exists():
            return q
    return None


ROOT = find_root()
if ROOT is not None:
    os.environ.setdefault("HF_HOME", str(ROOT / ".cache" / "hf"))
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")           # MPS에서 미지원 연산은 CPU로 대체
TARGETS = (ROOT / ".cache" / "enformer" / "targets_human.txt") if ROOT else Path("targets_human.txt")

import json
import time
import urllib.request

import matplotlib.pyplot as plt
import torch
import transformers
from enformer_pytorch import Enformer, str_to_one_hot

print("transformers", transformers.__version__, "| torch", torch.__version__)
assert transformers.__version__.startswith("4."), "격리 환경(.venv-enformer)에서 실행하세요 — 위 안내 참고"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEVICE)

## 1. 유전자 TSS와 196,608 bp 서열 가져오기
UCSC 유전체 브라우저 REST API (`api.genome.ucsc.edu`)에서 (1) hg38 RefSeq Select 주석으로 **GAPDH**의 전사 시작 좌표를, (2) 그 주변 서열을 조회한다.
GAPDH는 널리 발현되는 관리(housekeeping) 유전자라 대부분 세포에서 TSS 주변에 강한 프로모터 신호가 기대된다 — 정합성 점검에 쓰기 좋다.

In [ ]:
UCSC = "https://api.genome.ucsc.edu/getData"


def ucsc_get(url):
    with urllib.request.urlopen(url, timeout=60) as r:
        return json.load(r)


ann = ucsc_get(f"{UCSC}/track?genome=hg38;track=ncbiRefSeqSelect;chrom=chr12;start=6530000;end=6545000")["ncbiRefSeqSelect"]
gapdh = [g for g in ann if g["name2"] == "GAPDH"][0]
print({k: gapdh[k] for k in ["name", "name2", "chrom", "strand", "txStart", "txEnd"]})
assert gapdh["strand"] == "+", "이 노트북은 + 가닥 유전자를 가정"
TSS0 = gapdh["txStart"]                    # 0-based. UCSC txStart는 전사 시작 위치 (+ 가닥)
CHROM, L_IN = gapdh["chrom"], 196_608
START = TSS0 - L_IN // 2
CHUNK = 50_000
parts = []
for s in range(START, START + L_IN, CHUNK):
    e = min(s + CHUNK, START + L_IN)
    parts.append(ucsc_get(f"{UCSC}/sequence?genome=hg38;chrom={CHROM};start={s};end={e}")["dna"])
SEQ = "".join(parts).upper()
print(f"{CHROM}:{START:,}-{START + L_IN:,} ({len(SEQ):,} bp) | N 개수: {SEQ.count('N')} | TSS(0-based) {TSS0:,} 는 입력의 {TSS0 - START:,}번째 염기")
assert len(SEQ) == L_IN

## 2. 모델과 트랙 선택
트랙 메타데이터는 Calico Basenji 저장소의 `targets_human.txt`(5,313행)다. K562(만성골수성백혈병 세포주)의 트랙 몇 개를 고른다.

In [ ]:
model = Enformer.from_pretrained("EleutherAI/enformer-official-rough").to(DEVICE).eval()
print("params (M):", round(sum(p.numel() for p in model.parameters()) / 1e6))

TARGETS_URL = "https://raw.githubusercontent.com/calico/basenji/master/manuscripts/cross2020/targets_human.txt"      # 0.8 MB (Calico Basenji 저장소)
if not TARGETS.exists():
    TARGETS.parent.mkdir(parents=True, exist_ok=True)
    print("downloading", TARGETS_URL)
    urllib.request.urlretrieve(TARGETS_URL, TARGETS)
tg = pd.read_csv(TARGETS, sep="\t")
tg["kind"] = tg.description.str.split(":").str[0]
print(tg.kind.value_counts().to_dict(), "| 합계", len(tg))
pick = {}
pick["CAGE K562"] = int(tg[(tg.kind == "CAGE") & tg.description.str.contains("K562")].index[0])
pick["DNase K562"] = int(tg[tg.description == "DNASE:K562"].index[0])
pick["H3K4me3 K562"] = int(tg[tg.description == "CHIP:H3K4me3:K562"].index[0])
pick["H3K27me3 K562"] = int(tg[tg.description == "CHIP:H3K27me3:K562"].index[0])
pick["CTCF K562"] = int(tg[tg.description == "CHIP:CTCF:K562"].index[0])
for k, i in pick.items():
    print(f"{k:14s} 트랙 {i:5d} | {tg.description[i]}")

## 3. 예측
입력은 one-hot(A,C,G,T; N은 모두 0). 출력 `(1, 896, 5313)`은 **입력 중앙 114,688 bp**에 해당하는 128 bp bin별 예측이다.

In [ ]:
@torch.no_grad()
def predict(seq):
    x = str_to_one_hot([seq]).to(DEVICE)
    out = model(x, head="human")
    if DEVICE.type == "mps":
        torch.mps.synchronize()
    return out[0].float().cpu().numpy()                     # (896, 5313)


t0 = time.time()
pred = predict(SEQ)
print("출력 shape:", pred.shape, f"| {time.time() - t0:.1f}s (첫 호출)")
t0 = time.time()
pred = predict(SEQ)
T_FWD = time.time() - t0
print(f"두 번째 호출: {T_FWD:.1f}s")

N_BINS, BIN = pred.shape[0], 128
bin_of_tss = (TSS0 - START - (L_IN - N_BINS * BIN) // 2) // BIN         # TSS가 속하는 출력 bin
# 확인: TSS − START − 40,960(왼쪽 잘리는 부분) = 57,344 = 448 × 128, 나머지 0 → TSS는 bin 448의 **첫 번째 염기**다 (bin 448 = TSS 기준 [0, +128) bp).
print("TSS가 속하는 출력 bin:", bin_of_tss, "(전체 0~895, 중앙은 448)")

## 4. 트랙 그리기

In [ ]:
x_kb = (np.arange(N_BINS) * BIN + BIN / 2 + (L_IN - N_BINS * BIN) // 2 + START - TSS0) / 1000      # bin 중심 좌표 (TSS 기준)
fig, axes = plt.subplots(len(pick), 1, figsize=(9, 8), sharex=True)
for ax, (name, i) in zip(axes, pick.items()):
    ax.fill_between(x_kb, pred[:, i], step="mid", lw=0)
    ax.axvline(0, color="r", lw=0.8)
    ax.set_ylabel(name, rotation=0, ha="right", fontsize=8)
axes[-1].set_xlabel("TSS 기준 거리 (kb), 빨간 선 = GAPDH TSS")
axes[0].set_title(f"Enformer 예측: {CHROM}, GAPDH 주변")
plt.tight_layout()
plt.show()

## 5. 정합성 점검 — 숫자로 확인하기
(1) **TSS 근처 vs 평균**: 중앙 ±2 bin의 평균 예측 ÷ 전체 896 bin 평균 (K562 CAGE·DNase·H3K4me3).
(2) **음성 대조**: 같은 서열을 **무작위로 섞어**(염기 조성은 같고 구조는 파괴) 같은 점검을 반복 — 신호가 사라지는가?
(3) **전체 CAGE 트랙 638개**에서 TSS 근처 신호가 평균보다 높은 트랙의 비율.

In [ ]:
N_SHUF = 5
pred_shufs = [predict("".join(np.random.RandomState(sd).permutation(list(SEQ)))) for sd in range(N_SHUF)]      # 서로 다른 시드로 5번 섞음


def center_ratio(p, i, half=2):
    c = slice(bin_of_tss - half, bin_of_tss + half + 1)
    return p[c, i].mean() / (p[:, i].mean() + 1e-9)


print(f"{'트랙':14s}{'원래 서열':>12s}{'섞은 서열(5회 평균 ± SD)':>28s}")
for name, i in pick.items():
    if name in ("H3K27me3 K562", "CTCF K562"):
        continue
    r_s = [center_ratio(q, i) for q in pred_shufs]
    print(f"{name:14s}{center_ratio(pred, i):12.2f}{np.mean(r_s):17.2f} ± {np.std(r_s):.2f}")

cage_idx = tg.index[tg.kind == "CAGE"].values
r_orig = np.array([center_ratio(pred, i) for i in cage_idx])
r_shuf = np.array([[center_ratio(q, i) for i in cage_idx] for q in pred_shufs])        # (5, 638)
print(f"\nCAGE 트랙 {len(cage_idx)}개 전체에 대해 (TSS 근처 ÷ 평균):")
print(f"  '1보다 큰' 트랙 비율 — 원래 {np.mean(r_orig > 1):.2f} | 섞은 서열 {np.mean(r_shuf > 1):.2f}   ← 섞은 서열에서도 높다면 이 지표는 구분력이 없다")
print(f"  비율의 중앙값 — 원래 {np.median(r_orig):.1f} | 섞은 서열 {np.median(r_shuf):.1f} (5회: {np.round(np.median(r_shuf, axis=1), 1).tolist()})")
print(f"  원래 서열의 비율이 섞은 서열보다 큰 트랙 비율: {np.mean(r_orig[None] > r_shuf):.2f}")
all_c = slice(bin_of_tss - 2, bin_of_tss + 3)
prof = [float(np.median(q[all_c].mean(0) / (q.mean(0) + 1e-9))) for q in pred_shufs]
print(f"섞은 서열 입력에서 5,313개 전체 트랙의 중앙 비율 중앙값(5회): {np.round(prof, 2).tolist()}  ← 1보다 크면 서열 정보와 무관하게 중앙이 높게 예측되는 경향 (위치 편향 가설)")
print("\nISM에서 쓸 TSS 근처 서열(TSS-35 ~ TSS+10, 5'→3'):", SEQ[TSS0 - START - 35:TSS0 - START + 10])
peak_bin = bin_of_tss - 10 + int(np.argmax(pred[bin_of_tss - 10:bin_of_tss + 11, pick["CAGE K562"]]))
print(f"K562 CAGE 예측의 TSS ±10 bin 내 최대 위치: TSS에서 {(peak_bin - bin_of_tss) * BIN:+d} bp (bin 해상도 {BIN} bp)")

## 6. In silico mutagenesis (ISM)
서열의 한 염기를 다른 염기로 바꿔 K562 CAGE 예측(TSS ±2 bin 평균)이 얼마나 변하는지 본다.
- **가까운 위치**: TSS 근처 (TSS −30 ~ +30 bp, 5 bp 간격)
- **먼 위치**: TSS에서 +40 kb 떨어진 같은 개수의 위치 (출력 구간 안)
각 위치에서 나머지 3개 염기로 바꿔 예측하므로 forward가 (위치 수 × 3)번 필요하다.

In [ ]:
NEAR = np.arange(-30, 31, 5)
FAR = 40_000 + np.arange(-30, 31, 5)
cage_i = pick["CAGE K562"]
base_val = pred[bin_of_tss - 2:bin_of_tss + 3, cage_i].mean()


def ism(offsets):
    """열 = A, C, G, T 고정. ref 염기와 같은 칸은 NaN."""
    out = np.full((len(offsets), 4), np.nan)
    for r, off in enumerate(offsets):
        p = TSS0 - START + int(off)
        for c, a in enumerate("ACGT"):
            if a == SEQ[p]:
                continue
            q = predict(SEQ[:p] + a + SEQ[p + 1:])
            out[r, c] = q[bin_of_tss - 2:bin_of_tss + 3, cage_i].mean() - base_val
    return out


t0 = time.time()
d_near = ism(NEAR)
d_far = ism(FAR)
print(f"ISM 완료: 예측 {2 * len(NEAR) * 3}회, {time.time() - t0:.0f}s (예측 1회 약 {T_FWD:.1f}s)")
print(f"기준 예측값(K562 CAGE, TSS ±2 bin 평균): {base_val:.3f}")
print(f"|Δ| 평균 — 가까운 위치(±30 bp): {np.nanmean(np.abs(d_near)):.4f} | 먼 위치(+40 kb): {np.nanmean(np.abs(d_far)):.4f} | 비율 {np.nanmean(np.abs(d_near)) / (np.nanmean(np.abs(d_far)) + 1e-12):.1f}배")
r_max = np.unravel_index(np.nanargmax(np.abs(d_near)), d_near.shape)
print(f"|Δ| 최대 — 가까운 위치: {np.nanmax(np.abs(d_near)):.4f} (TSS{NEAR[r_max[0]]:+d} bp, {SEQ[TSS0 - START + int(NEAR[r_max[0]])]}→{'ACGT'[r_max[1]]}) | 먼 위치: {np.nanmax(np.abs(d_far)):.4f}")

In [ ]:
vmax = np.nanmax(np.abs(d_near))
fig, ax = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
for a, d, offs, title in [(ax[0], d_near, NEAR, "TSS 근처 (±30 bp)"), (ax[1], d_far, FAR, "먼 위치 (+40 kb)")]:
    im = a.imshow(np.ma.masked_invalid(d.T), aspect="auto", cmap="coolwarm", vmin=-vmax, vmax=vmax)
    a.set_xticks(range(len(offs)))
    a.set_xticklabels([f"{o}\n{SEQ[TSS0 - START + int(o)]}" for o in offs], fontsize=7)       # 위치와 ref 염기
    a.set_yticks(range(4))
    a.set_yticklabels(list("ACGT"))
    a.set_title(title, fontsize=9)
    a.set_xlabel("TSS 기준 위치 (bp) / ref 염기")
ax[0].set_ylabel("alt 염기 (ref와 같은 칸은 비워 둠)")
plt.colorbar(im, ax=ax, fraction=0.03)
plt.show()

## 7. 해석 가이드와 한계
이 실행(MPS)에서 읽은 내용이다. **숫자는 직접 다시 확인하라.**

- **TSS 신호**: GAPDH TSS 주변 K562 CAGE 예측은 전체 평균의 약 62배, DNase는 약 30배, H3K4me3는 약 6배다. 예측된 CAGE 최대 위치는 TSS가 속한 128 bp bin이다. 그림에서 TSS의 빨간 선 위치에 봉우리가 있는지 눈으로도 확인하라.
- **음성 대조가 꼭 필요하고, 한 번의 shuffle로 결론 내면 안 된다**: 서열을 5번 섞으면 같은 비율이 CAGE 1.6 ± 0.8배, DNase 1.4 ± 0.4배, H3K4me3 1.2 ± 0.3배로 크게 줄어든다. CAGE 638개 트랙의 중앙값은 원래 서열 70, 섞은 서열 0.8~2.5(시도마다 다름)이고 **모든 트랙에서 원래 서열이 섞은 서열보다 크다**.
  "TSS 근처가 평균보다 높은 CAGE 트랙의 비율"은 원래 1.00, 섞은 서열 5회 평균 0.71로 **지표로서 구분력이 약하다** — 첫 shuffle 하나만 봤을 때는 이 비율이 1.00이어서 "모델의 중앙 위치 편향"으로 해석하고 싶어지지만 5회로 늘리자 그 해석이 흔들렸다.
- **중앙 위치 편향은 가설일 뿐이다**: 섞은 서열에서 5,313개 전체 트랙의 중앙 비율 중앙값이 5회 중 4회는 1보다 컸지만(1.18~1.59) 한 번은 0.72였다. 서열 정보와 무관한 중앙 상승 경향이 있는지는 이 실험으로 결론낼 수 없다 — 더 많은 shuffle과 대조 위치(예: TSS가 없는 임의 위치)가 필요하다.
- **ISM**: 이 설정에서 K562 CAGE 예측(TSS ±2 bin 평균, 기준값 약 152)에 대한 |Δ| 평균은 TSS 근처(±30 bp)가 약 2.6, +40 kb 떨어진 위치가 약 0.06으로 약 46배 차이다. 가장 큰 변화는 TSS −30 bp의 C→A에서 약 14.6(기준값의 약 10%)이었다.
  위에 출력한 서열의 TSS −29~−23에는 TATA 박스와 비슷한 `TATAAAT`이 있고, 가장 큰 변화(−30 bp)는 이 요소 바로 앞쪽에서 나타났다. **이는 관찰일 뿐**이다 — 모티프 판정은 모티프 DB로, 효과의 실제 여부는 실험으로 확인해야 하며 이 노트북은 검증하지 않았다 (과제 4).
- 예측 1회에 약 6초(MPS)가 걸려 ISM 78회에 약 8분이 걸렸다.

**한계**
- 유전자 하나(GAPDH), 세포주 하나(K562), 변이 몇 개로 본 정합성 점검이다. 모델의 정확도를 평가한 것이 아니며, **실측 트랙이나 독립적인 변이 실험과 비교하지 않았다**.
- GAPDH는 TSS 근처에 TATA 박스 유사 서열을 가진 사례다. 포유류 프로모터에는 TATA 박스가 없는 경우(CpG 섬 프로모터 등)가 많고 그런 프로모터에서는 ISM 민감도가 몇 개 염기에 집중되지 않고 더 넓게 퍼질 수 있다† — 과제 4에서 다른 유전자와 비교하라.
- Enformer는 **참조 게놈**의 유전자 간·세포 유형 간 차이를 잘 학습하지만, **개인 간 cis 변이에 의한 발현 차이**는 잘 설명하지 못한다는 보고가 있다 (Huang et al. 2023, 강의노트 6절).
- ISM의 ΔCAGE는 예측 변화일 뿐 실험으로 확인된 효과가 아니다.
- **입력 범위(196,608 bp)와 출력 범위(중앙 896 bins ≈ 114,688 bp)는 다르다.** 입력 안의 변이는 모델의 수용장을 통해 중앙 출력에 영향을 줄 수 있다. 다만 출력 구간 **밖** 위치의 트랙은 이 호출로 직접 예측되지 않으므로, 변이 효과를 보려면 변이 주변의 출력 bin을 읽을 수 있게 변이를 출력 구간 안에 두는 창을 쓴다.
- 과제: [../exercises.md](../exercises.md) 과제 4.